In [100]:
import pandas as pd
import plotly.express as px

import calliope

# We increase logging verbosity
calliope.set_log_verbosity("INFO", include_solver_output=False)

In [101]:
model = calliope.read_yaml('model.yaml')

[2026-03-16 12:51:41] INFO     Math init | loading pre-defined math.
[2026-03-16 12:51:41] INFO     Math init | loading math files {'base', 'operate', 'milp', 'spores', 'storage_inter_cluster'}.
[2026-03-16 12:51:41] INFO     Model: preprocessing data
[2026-03-16 12:51:41] INFO     Math build | building applied math with ['base'].
[2026-03-16 12:51:41] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2026-03-16 12:51:41] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[2026-03-16 12:51:41] INFO     input data `cost_source_use` not defined in model math; it will not be available in the optimisation problem.
[2026-03-16 12:51:41] INFO     input data `color` not defined in model math; it will not be available in the optimisation problem.
[2026-03-16 12:51:41] INFO     input data `name` not defined in model math; it will not be available in the optimisation problem.
[202

In [90]:
model.inputs

<xarray.Dataset> Size: 10kB
Dimensions:                     (costs: 1, techs: 5, nodes: 3, carriers: 1,
                                 timesteps: 48)
Coordinates:
  * costs                       (costs) object 8B 'monetary'
  * techs                       (techs) object 40B 'X1_to_X2' ... 'supply_gri...
  * carriers                    (carriers) object 8B 'electricity'
  * nodes                       (nodes) object 24B 'X1' 'X2' 'X3'
  * timesteps                   (timesteps) datetime64[ns] 384B 2005-07-01 .....
Data variables: (12/32)
    cost_interest_rate          (costs) float64 8B 0.1
    bigM                        float64 8B 1e+06
    objective_cost_weights      (costs) float64 8B 1.0
    base_tech                   (techs) object 40B 'transmission' ... 'supply'
    carrier_in                  (nodes, techs, carriers) bool 15B True ... False
    color                       (techs) object 40B '#6783E3' ... '#C5ABE3'
    ...                          ...
    longitude                   (nodes) float64 24B -0.1613 -0.1142 -0.1311
    source_use_equals           (techs, timesteps) float64 2kB nan nan ... nan
    sink_use_equals             (timesteps, techs, nodes) float64 6kB nan ......
    definition_matrix           (nodes, techs, carriers) bool 15B True ... False
    timestep_resolution         (timesteps) float64 384B 1.0 1.0 1.0 ... 1.0 1.0
    timestep_weights            (timesteps) float64 384B 1.0 1.0 1.0 ... 1.0 1.0

In [91]:
model.inputs.flow_cap_max.to_series().dropna()

techs              nodes
X1_to_X2           X1       2000.0
                   X2       2000.0
X1_to_X3           X1       2000.0
                   X3       2000.0
pv                 X1        250.0
                   X2        250.0
                   X3         50.0
supply_grid_power  X1       2000.0
Name: flow_cap_max, dtype: float64

In [92]:
model.inputs.sink_use_equals.sum(
    "timesteps", min_count=1, skipna=True
).to_series().dropna()

techs               nodes
demand_electricity  X1         35.271156
                    X2       8796.878622
                    X3       1244.604116
Name: sink_use_equals, dtype: float64

In [93]:
model.build(force=True)
model.solve(solver='gurobi')

[2026-03-16 11:50:21] INFO     Model: backend build starting
[2026-03-16 11:50:21] INFO     Optimisation Model | parameters/lookups | Generated.
[2026-03-16 11:50:21] INFO     Optimisation Model | variables | Generated.
[2026-03-16 11:50:23] INFO     Optimisation Model | global_expressions | Generated.
[2026-03-16 11:50:24] INFO     Optimisation Model | constraints | Generated.
[2026-03-16 11:50:24] INFO     Optimisation Model | piecewise_constraints | Generated.
[2026-03-16 11:50:24] INFO     Optimisation Model | objectives | Generated.
[2026-03-16 11:50:24] INFO     Model: backend build complete
[2026-03-16 11:50:24] INFO     Optimisation model | starting model in base mode.
[2026-03-16 11:50:24] INFO     Backend: solver finished running. Time since start of solving optimisation problem: 0:00:00.145106
[2026-03-16 11:50:24] INFO     Postprocessing: applied zero threshold 1e-10 to model results.
[2026-03-16 11:50:24] INFO     Postprocessing: ended. Time since start of solving optimisa

In [94]:
model.results

<xarray.Dataset> Size: 51kB
Dimensions:                     (nodes: 3, techs: 5, carriers: 1,
                                 timesteps: 48, costs: 1)
Coordinates:
  * techs                       (techs) object 40B 'X1_to_X2' ... 'supply_gri...
  * nodes                       (nodes) object 24B 'X1' 'X2' 'X3'
  * carriers                    (carriers) object 8B 'electricity'
  * timesteps                   (timesteps) datetime64[ns] 384B 2005-07-01 .....
  * costs                       (costs) object 8B 'monetary'
Data variables: (12/24)
    flow_cap                    (nodes, techs, carriers) float64 120B 273.9 ....
    link_flow_cap               (techs) float64 40B 273.9 45.78 nan nan nan
    flow_out                    (nodes, techs, carriers, timesteps) float64 6kB ...
    flow_in                     (nodes, techs, carriers, timesteps) float64 6kB ...
    flow_export                 (nodes, techs, carriers, timesteps) float64 6kB ...
    area_use                    (nodes, techs) float64 120B nan nan ... nan
    ...                          ...
    min_cost_optimisation       float64 8B -57.98
    capacity_factor             (nodes, techs, carriers, timesteps) float64 6kB ...
    systemwide_capacity_factor  (techs, carriers) float64 40B 0.3345 ... 0.6976
    systemwide_levelised_cost   (techs, costs, carriers) float64 40B 1.88e-06...
    total_levelised_cost        (costs, carriers) float64 8B -0.004148
    unmet_sum                   (nodes, carriers, timesteps) float64 1kB 0.0 ...

In [95]:
costs = model.results.cost.to_series().dropna()
costs.head()

nodes  techs              costs   
X1     X1_to_X2           monetary     0.008266
       X1_to_X3           monetary     0.000691
       pv                 monetary     0.000000
       supply_grid_power  monetary    17.743388
X2     X1_to_X2           monetary     0.008266
Name: cost, dtype: float64

In [96]:
lcoes = (
    model.results.systemwide_levelised_cost.sel(carriers="electricity")
    .to_series()
    .dropna()
)
lcoes.head()

techs              costs   
X1_to_X2           monetary    0.000002
X1_to_X3           monetary    0.000002
pv                 monetary   -0.018309
supply_grid_power  monetary    0.001803
Name: systemwide_levelised_cost, dtype: float64

In [97]:
# We set the color mapping to use in all our plots by extracting the colors defined in the technology definitions of our model.
colors = model.inputs.color.to_series().to_dict()

In [98]:
df_electricity = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers="electricity")
    .sum("nodes")
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_electricity_demand = df_electricity[df_electricity.techs == "demand_electricity"]
df_electricity_other = df_electricity[df_electricity.techs != "demand_electricity"]

print(df_electricity.head())

fig1 = px.bar(
    df_electricity_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    color="techs",
    color_discrete_map=colors,
)
fig1.add_scatter(
    x=df_electricity_demand.timesteps,
    y=-1 * df_electricity_demand["Flow in/out (kWh)"],
    marker_color="black",
    name="demand",
)

      techs           timesteps  Flow in/out (kWh)
0  X1_to_X2 2005-07-01 00:00:00          -1.929506
1  X1_to_X2 2005-07-01 01:00:00          -1.570625
2  X1_to_X2 2005-07-01 02:00:00          -1.581138
3  X1_to_X2 2005-07-01 03:00:00          -1.581138
4  X1_to_X2 2005-07-01 04:00:00          -1.688398


In [99]:
carriers = ["electricity"]
df_flows = (
    (model.results.flow_out.fillna(0) - model.results.flow_in.fillna(0))
    .sel(carriers=carriers)
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow in/out (kWh)")
    .reset_index()
)
df_demand = df_flows[df_flows.techs.str.contains("demand")]
df_flows_other = df_flows[~df_flows.techs.str.contains("demand")]

print(df_flows.head())

node_order = df_flows_other.nodes.unique()

fig = px.bar(
    df_flows_other,
    x="timesteps",
    y="Flow in/out (kWh)",
    facet_row="nodes",
    facet_col="carriers",
    color="techs",
    category_orders={"nodes": node_order, "carriers": carriers},
    height=1000,
    color_discrete_map=colors,
)

showlegend = True
# we reverse the node order (`[::-1]`) because the rows are numbered from bottom to top.
for row, node in enumerate(node_order[::-1]):
    for col, carrier in enumerate(carriers):
        demand_ = df_demand.loc[
            (df_demand.nodes == node) & (df_demand.techs == f"demand_{carrier}"),
            "Flow in/out (kWh)",
        ]
        if not demand_.empty:
            fig.add_scatter(
                x=model.results.timesteps.values,
                y=-1 * demand_,
                row=row + 1,
                col=col + 1,
                marker_color="black",
                name="Demand",
                legendgroup="demand",
                showlegend=showlegend,
            )
            showlegend = False
fig.update_yaxes(matches=None)
fig.show()

  nodes     techs     carriers           timesteps  Flow in/out (kWh)
0    X1  X1_to_X2  electricity 2005-07-01 00:00:00         -96.475307
1    X1  X1_to_X2  electricity 2005-07-01 01:00:00         -78.531244
2    X1  X1_to_X2  electricity 2005-07-01 02:00:00         -79.056888
3    X1  X1_to_X2  electricity 2005-07-01 03:00:00         -79.056888
4    X1  X1_to_X2  electricity 2005-07-01 04:00:00         -84.419893


In [88]:
df_capacity = (
    model.results.flow_cap.where(
        ~model.inputs.base_tech.str.contains("demand|transmission")
    )
    .to_series()
    .where(lambda x: x != 0)
    .dropna()
    .to_frame("Flow capacity (kW)")
    .reset_index()
)

print(df_capacity.head())

fig = px.bar(
    df_capacity,
    x="nodes",
    y="Flow capacity (kW)",
    color="techs",
    facet_col="carriers",
    color_discrete_map=colors,
)
fig.show()

  nodes              techs     carriers  Flow capacity (kW)
0    X1  supply_grid_power  electricity           314.90168
